# Set up

In [ ]:
import os
import json


def load_json(path):
    with open(os.path.join(path), "r") as f:
        return json.load(f)

def save_json(data, filename):
    with open(os.path.join("json", filename), "w") as f:
        json.dump(data, f, indent=4)


# Google Drive Client


In [ ]:
from googleapiclient.discovery import build
import requests


class GoogleDriveClient:
    def __init__(self, api_key, output_path: str = "output"):
        self.service = build("drive", "v3", developerKey=api_key)
        self.output_path = output_path

    def download_file(self, file_id):
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        r = requests.get(url, stream=True)
        with open(os.path.join(self.output_path, f"{file_id}"), "wb") as f:
            for chunk in r.iter_content(1024):
                if chunk:
                    f.write(chunk)

    def list_subfolders(self, folder_id):
        query = f"'{folder_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
        res = self.service.files().list(q=query, fields="files(id, name)").execute()
        return res.get("files", [])

    def list_files(self, folder_id):
        query = f"'{folder_id}' in parents and trashed=false"
        res = (
            self.service.files()
            .list(q=query, fields="files(id, name, mimeType)")
            .execute()
        )
        return res.get("files", [])

# Snapshot client

In [ ]:
import aiohttp
import certifi
import ssl
from yarl import URL


class SnapshotClient:
    def __init__(self) -> None:
        self.ssl_context = ssl.create_default_context(cafile=certifi.where())
        self.connector = aiohttp.TCPConnector(ssl=self.ssl_context)
        self.session: aiohttp.ClientSession | None = None
        self.url = "https://app.fotomonitoreo.cl/"
        self.origin = "https://app.fotomonitoreo.cl"
        self.headers = None

    async def __aenter__(self):
        if self.session is None or self.session.closed:
            self.session = aiohttp.ClientSession(connector=self.connector)
        await self._get_headers()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        if self.session is not None and not self.session.closed:
            await self.session.close()
        if not self.connector.closed:
            await self.connector.close()

    async def _ensure_session(self) -> aiohttp.ClientSession:
        if self.session is None or self.session.closed:
            self.session = aiohttp.ClientSession(connector=self.connector)
        return self.session

    async def _get_headers(self) -> dict:
        session = await self._ensure_session()
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Referer": self.url,
            "Origin": self.origin,
            "X-Requested-With": "XMLHttpRequest",
        }
        async with session.get(self.url, headers=headers) as response:
            await response.text()
        csrf_token = session.cookie_jar.filter_cookies(URL(self.url)).get("csrftoken")
        csrf_value = csrf_token.value if csrf_token is not None else None
        headers = {
            **headers,
            "X-CSRFToken": csrf_value,
            "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        }
        self.headers = headers
        return headers

    async def _request_json(self, method: str, path: str, data: dict | None = None):
        session = await self._ensure_session()
        if self.headers is None:
            await self._get_headers()

        async with session.request(
            method,
            f"{self.url}{path}",
            headers=self.headers,
            data=data,
        ) as response:
            response.raise_for_status()
            return await response.json()

    async def regiones(self) -> dict:
        try:
            return await self._request_json("GET", "visor/regiones/")
        except aiohttp.ClientError as e:
            print(f"Error fetching regions: {e}")
            return {}

    async def unidades_by_region(self, codigo_region: int) -> dict:
        try:
            data = {
                "codigo_region": int(codigo_region),
            }
            return await self._request_json("POST", "visor/unidades_by_region/", data)
        except aiohttp.ClientError as e:
            print(f"Error fetching unidades for region {codigo_region}: {e}")
            return {}

    async def year_by_unidad(self, codigo_unidad: str):
        try:
            data = {
                "codigo_unidad": codigo_unidad,
            }
            return await self._request_json("POST", "visor/year_by_unidad/", data)
        except aiohttp.ClientError as e:
            print(f"Error fetching years for unidad {codigo_unidad}: {e}")
            return {}

    async def especie_by_unidadyear(self, codigo_unidad: str, year: int):
        try:
            data = {
                "codigo_unidad": codigo_unidad,
                "year": year,
            }
            return await self._request_json(
                "POST", "visor/especie_by_unidadyear/", data
            )
        except aiohttp.ClientError as e:
            print(
                f"Error fetching especies for unidad {codigo_unidad} and year {year}: {e}"
            )
            return {}

    async def grillas_by_year_geojson(self, codigo_unidad: str, year: int):
        try:
            data = {
                "codigo_unidad": codigo_unidad,
                "year": year,
            }
            return await self._request_json(
                "POST", "visor/grillas_by_year_geojson/", data
            )
        except aiohttp.ClientError as e:
            print(
                f"Error fetching grillas for unidad {codigo_unidad} and year {year}: {e}"
            )
            return {}

    async def grillas_by_year_especie_geojson(
        self, codigo_unidad: str, year: int, especie: str, only_urls: bool = False
    ):
        try:
            data = {
                "codigo_unidad": codigo_unidad,
                "year": year,
                "especie": especie,
            }
            response = await self._request_json(
                "POST", "visor/grillas_by_year_especie_geojson/", data
            )
            if not only_urls:
                return response
            return list(
                map(lambda f: f["properties"]["gdrive_url"], response["features"])
            )
        except aiohttp.ClientError as e:
            print(
                f"Error fetching grillas for unidad {codigo_unidad}, year {year} and especie {especie}: {e}"
            )
            return {}

In [ ]:
"""
Json Table with species information.
It's includes region code -> park code -> year -> specie code -> [urls]
"""

REGIONES = [x for x in range(7, 17)]
RETRIEVE_URLS = True

sp_client = SnapshotClient()
regiones = await sp_client.regiones()
regiones_filtradas = [
    r
    for r in regiones["regiones"]
    if r["codigo"] in REGIONES
]

table = {}
table_path = os.path.join('/content', 'data')
if not os.path.exists(table_path):
    os.mkdir(table_path)

if RETRIEVE_URLS:
    table_json_file = os.path.join(table_path, "tabla.json")
    for region in regiones_filtradas:
        codigo_region = region["codigo"]
        print("Checking region: ", codigo_region)
        table[codigo_region] = {}
        unidades = await sp_client.unidades_by_region(codigo_region)
        for unidad in unidades["unidades"]:
            codigo_unidad = unidad["codigo"]
            table[codigo_region][codigo_unidad] = {}
            years = await sp_client.year_by_unidad(codigo_unidad)
            for year in years["years"]:
                year_value = year["year"]
                table[codigo_region][codigo_unidad][year_value] = {}
                especies = await sp_client.especie_by_unidadyear(
                    codigo_unidad, year_value
                )
                for especie in especies["especies"]:
                    codigo_especie = especie["codigo"]
                    urls = await sp_client.grillas_by_year_especie_geojson(
                        codigo_unidad, year_value, codigo_especie, only_urls=True
                    )
                    table[codigo_region][codigo_unidad][year_value][codigo_especie] = (
                        urls
                    )
    save_json(table, table_json_file)
else:
    try:
      table = load_json(table_json_file)
    except FileNotFoundError:
      print("Table not found")

"""
Flatmap of specie code and urls
"""
REGIONS = table.keys()
species_images = {}
# Must run this code after the table is already loaded!
async with SnapshotClient() as snapshot_client:
  for region_code in REGIONS:
    parks = await snapshot_client.unidades_by_region(region_code)
    parks_codes = [park['codigo'] for park in parks['unidades']]

    for park_code in parks_codes:
      years_by_park = await snapshot_client.year_by_unidad(park_code)
      years_values = [str(item['year']) for item in years_by_park['years']]

      for year in years_values:
        try:
          species = table[region_code][park_code][int(year)]
        except KeyError:
          species = table[region_code][park_code][year]
        for specie_code, urls in species.items():
          if not species_images.get(specie_code, None):
            species_images[specie_code] = []
          species_images[specie_code].extend(urls)


ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7a7628737c50>


Checking region:  7
Checking region:  8
Checking region:  9
Checking region:  10
Checking region:  11
Checking region:  12
Checking region:  13
Checking region:  14
Checking region:  15
Checking region:  16


In [ ]:
species_images.keys()

dict_keys(['EQCA', 'ORCU', 'LEER', 'CAFA', 'PUCO', 'GACU', 'BOTA', 'LAWO', 'LYCU', 'LECO', 'FECA', 'LAGU', 'CAHI', 'COCH', 'LEGU', 'PTTA', 'EQAS', 'OVAR', 'SCRU', 'HIBI', 'PUPU', 'LYSP', 'ARCR', 'SUSC', 'NEVI', 'LYGR', 'SUDO', 'MYCO', 'LYFU', 'LOPR', 'LEGE', 'CHVI', 'COHU', 'CEEL'])

# Species codes

In [ ]:
SPECIE_CODE_TO_NAME = {
    "EQCA": ["Caballo", "caballo"],
    "COCH": ["Chingue", "chingue"],
    "LEGE": ["Gato de Geoffroy", "geoffroy", "Geoffroy"],
    "LAGU": ["Guanaco", "guanaco"],
    "HIBI": ["Huemul", "huemul"],
    "LEER": ["Liebre europea", "liebre", "Liebre"],
    "OVAR": ["Oveja", "oveja"],
    "CAFA": ["Perro domestico", "perro", "Perro"],
    "PUCO": ["Puma", "puma"],
    "BOTA": ["Vaca", "vaca"],
    "LYCU": ["Zorro culpeo", "culpeo", "Culpeo"],
    "LECO": ["Gato colocolo", "colocolo", "Colocolo"],
    "NEVI": ["Vison americano", "vison", "Vison"],
    "LAWO": ["Vizcacha", "vizcacha"],
    'CHVI': ["Armadillo peludo", "quirquincho", "Quirquincho"],
    'GACU': ["Quique", "quique"],
    'COHU': ["Chingue patagonico", "chinguepatagonico"],
    'PTTA': ["Hued hued", "huedhueddelsur", "huedhued", "hued hued"],
    'SUSC': ["Jabali", "jabali"],
    'LYGR': ["Zorro chilla", "Chilla", "chilla"],
    'LEGU': ["guina", "guigna"], # "Guina", "guina",
    'CEEL': ["Ciervo rojo", "Ciervorojo"],
    'LYSP': ["Zorro", "Zorrosp"],
    'ORCU': ["Conejo europeo", "liebre", "Liebre"],
    'MYCO': ["Coipo", "coipo"],
    'FECA': ["Gato domestico", "gato"],
    'SCRU': ["Chucao", "chucao"],
    "PUPU": ["Pudu", "pudu", "pudú"],
    "ZRZC": ["Zorzal", "zorzal"] # MANUAL

}

In [ ]:
import requests
def download_file(output_path, file_id, file_name): # Add file_name parameter
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    r = requests.get(url, stream=True)
    with open(os.path.join(output_path, f"{file_name}.png"), "wb") as f:
        for chunk in r.iter_content(1024):
            if chunk:
                f.write(chunk)

# Auto download

In [ ]:
DATASET_PATH = os.path.join(
    "/content", "drive", "MyDrive", "ECHO", "Data", "Training"
)
def load_json(path):
    with open(os.path.join(path), "r") as f:
        return json.load(f)

def save_json(data, path):
    with open(os.path.join(path), "w") as f:
        json.dump(data, f, indent=4)


In [ ]:
detections_metadata = load_json(os.path.join(DATASET_PATH, "detections_metadata.json"))

id_to_specie = {}
for cls, values in detections_metadata.items():
    first_key = next(iter(values))
    cls_name = values[first_key]["class_name"].lower().replace(" ", "_")
    id_to_specie[cls] = cls_name

specie_to_id = {v : k for k,v in id_to_specie.items()}

In [ ]:
len(list(detections_metadata[specie_to_id["guina"]].keys()))

314

In [ ]:
API_KEY = "" 
google_client =  GoogleDriveClient(API_KEY, DATASET_PATH)
BASE_PATH = "/content/data"
# Species to download
TO_DOWNLOAD = ["LEGU"]
restore = {}
counter = {specie : 0 for specie in TO_DOWNLOAD}
for specie_code in counter:
    name = SPECIE_CODE_TO_NAME[specie_code][0].lower()
    specie_path = os.path.join(BASE_PATH,  specie_to_id[name])
    restore[specie_code] = {'name': name, 'id': specie_to_id[name], 'original': [], 'final': [], "new": []}

    if not os.path.exists(specie_path):
        os.makedirs(specie_path, exist_ok=True)

    # Restore metadata detections
    for img_name, img_dict in detections_metadata[specie_to_id[name]].items():
        clean_img_name = img_name.replace(".png", "")
        dest_img = img_dict["dest_img_path"].split("/")[-1]
        dest_path = os.path.join(BASE_PATH, specie_to_id[name], dest_img)
        restore[specie_code]['original'].append(clean_img_name)
        restore[specie_code]['final'].append(dest_img)



In [ ]:
# Limit of files to download
LIMIT = 30

for specie_code in TO_DOWNLOAD:
    specie_name = restore[specie_code]['name']
    specie_id = restore[specie_code]['id']
    print(f"Downloading {specie_name} ({specie_id})")
    old_original_imgs = restore[specie_code]['original'] # This is the current files we have to compare.
    old_final_imgs = restore[specie_code]['final']

    specie_folder_ids = [
        item.split("/")[-1]
        for item in species_images[specie_code]
        if item != None
    ]

    counter = 0

    for specie_folder_id in specie_folder_ids:
        files = google_client.list_files(specie_folder_id)

        for f in files:
            fname = f["name"]

            if fname in SPECIE_CODE_TO_NAME[specie_code]:
                print("." * 2, SPECIE_CODE_TO_NAME[specie_code][0], fname)
                fid = f["id"]
                print("." * 2, f"{fid}/")
                subfolders = google_client.list_subfolders(fid)

                for subfolder in subfolders:
                    subfolder_name = subfolder["name"]
                    subfolder_id = subfolder["id"]
                    subfolder_files = [
                        (subfolder_file['id'], subfolder_file['name'])
                        for subfolder_file in google_client.list_files(subfolder_id)
                    ]

                    google_client.output_path = os.path.join(BASE_PATH, specie_id)

                    print("." * 2, f"Current counter {counter}")
                    print("." * 4,f"{subfolder_name}/")



                    for subfolder_file_id, subfolder_file_name in subfolder_files:
                        # Replace .png or .JPG or .AVI to .jpg
                        subfolder_file_name = subfolder_file_name.replace(".png", "")
                        subfolder_file_name = subfolder_file_name.replace(".JPG", "")
                        subfolder_file_name = subfolder_file_name.replace(".AVI", "")


                        #if not os.path.exists(os.path.join(google_client.output_path)):
                        #    os.makedirs(os.path.join(google_client.output_path), exist_ok=True)
                        if f"{subfolder_file_name}.jpg" in old_original_imgs or f"{subfolder_file_name}.JPG":
                            print("." * 6, f"{subfolder_file_name} already exists")
                            continue

                        download_file(google_client.output_path, subfolder_file_id, subfolder_file_name)
                        print("." * 6, f"{subfolder_file_name}")
                        restore[specie_code]['new'].append(subfolder_file_name)
                        counter += 1

                        if counter >= LIMIT:
                            break
                    if counter >= LIMIT:
                        break
                if counter >= LIMIT:
                    break




Se truncaron las últimas líneas 5000 del resultado de transmisión.
...... 2021 09 04 15 00 52 already exists
...... 2021 09 04 15 00 58 already exists
...... 2021 09 04 15 00 54 already exists
...... 2021 09 04 15 00 55 already exists
...... 2021 09 04 15 00 51 already exists
...... 2021 09 04 15 00 59 already exists
.. guina guigna
.. 1uXAxVZeWg9BI6s5obvIvG1RQOUmtYaPt/
.. Current counter 0
.... 1/
...... 2021 09 18 10 40 04 already exists
...... 2021 09 18 10 40 05 already exists
...... 2021 09 18 10 40 03 already exists
...... 2021 09 01 16 22 00 already exists
...... 2021 09 01 16 22 01 already exists
...... 2021 09 01 16 22 04 already exists
...... 2021 09 01 16 06 28 already exists
...... 2021 09 01 16 06 27 already exists
...... 2021 09 01 16 06 29 already exists
...... 2021 08 31 06 48 14 already exists
...... 2021 08 31 06 48 12 already exists
...... 2021 08 31 06 48 13 already exists
...... 2021 08 28 00 54 42 already exists
...... 2021 08 28 00 54 41 already exists
...... 202

In [ ]:
"""
# Descomentar si es necesario limitar la cantidad de archivos.
## Keep 500 files
from random import choice
for dir in os.listdir(MAIN_DIR):
  dir_path = os.path.join(MAIN_DIR, dir)
  files = os.listdir(dir_path)
  total_files = len(files)
  while total_files > 500:
    file_to_delete = choice(files)
    os.remove(os.path.join(dir_path, file_to_delete))
    total_files = len(os.listdir(dir_path))
    files = os.listdir(dir_path)
    print(f"Deleted {file_to_delete} files from {dir}. Total {total_files}")

"""

# Manual download for non-registered species


In [ ]:
import os
#API_KEY = ""
EXPORT_FILES = os.path.join("/content", "data", "test")
google_client =  GoogleDriveClient(API_KEY, 'data')

specie_to_find_folder_ids = set()
search_key = "aves"
for region in table:
    if region != 16:
        continue
    for park in table[region]:
        for year in table[region][park]:
            for specie in table[region][park][year]:
                urls = table[region][park][year][specie]
                if None in urls:
                    continue
                for url in urls:
                    folder_id = url.split("/")[-1]
                    folders = google_client.list_subfolders(folder_id)
                    folders_names = [folder["name"] for folder in folders]
                    if folder_id in specie_to_find_folder_ids:
                        continue
                    if search_key in folders_names:
                        print(region, year, specie, url)
                        specie_to_find_folder_ids.add(folder_id)
print(specie_to_find_folder_ids)


16 2022 EQCA https://drive.google.com/drive/folders/1BkNszkPyO3La1q1aZFWFRN7IqA2V9si8
16 2022 EQCA https://drive.google.com/drive/folders/1Bv62i0XGnNYlRtQpvID7lcKuQAapA2sj
16 2022 EQCA https://drive.google.com/drive/folders/14bCoqml442OJl2nuun_GuWtg2eW-9aEo
16 2022 EQCA https://drive.google.com/drive/folders/1E62HlUHE7oJla8-LCo9BEPWpHd5O6QUf
16 2022 COCH https://drive.google.com/drive/folders/14qfbwKA5GhzxhYqE7Q6sAMDi7haO03DP
16 2022 MYCO https://drive.google.com/drive/folders/158NVUbSzs1mPrsOB8v430fjdr8r35hU4
16 2022 LEGE https://drive.google.com/drive/folders/1SbPG6d9ES2KFBc3wLp9qtHkGQbE5-mGK
16 2022 LAGU https://drive.google.com/drive/folders/1AYdZiMX9nkvkncFCRR56uvYZjjr1qiDu
16 2022 LAGU https://drive.google.com/drive/folders/1Ai_OBRmTH8GDhllst9cAeIujtSCAb4bi
16 2022 LAGU https://drive.google.com/drive/folders/1AkTHtH8JPDQX6dALeK3QeWThtHn8nPo7
16 2022 LAGU https://drive.google.com/drive/folders/1ApxbHCfNETXSzb_ZdpGCaZk3BWvyCFHy
16 2022 LAGU https://drive.google.com/drive/folders/1A

dict_keys([7, 8, 9, 10, 11, 12, 13, 14, 15, 16])

In [ ]:
if not os.path.exists(f"/content/data/{search_key}"):
    os.makedirs(f"/content/data/{search_key}")
limit = 500
count = 0
for specie_folder_id in specie_folder_ids:
    files = google_client.list_files(specie_folder_id)
    for f in files:
        fname = f["name"]
        if fname == search_key:
            print("." * 2, f"{fname}/")
            subfolders = google_client.list_subfolders(f["id"])
            for subfolder in subfolders:
                subfolder_name = subfolder["name"]
                subfolder_id = subfolder["id"]
                subfolder_files = [
                    (subfolder_file['id'], subfolder_file['name'])
                    for subfolder_file in google_client.list_files(subfolder_id)
                ]
                for subfolder_file_id, subfolder_file_name in subfolder_files:
                    subfolder_file_name = subfolder_file_name.replace(".png", "")
                    subfolder_file_name = subfolder_file_name.replace(".JPG", "")
                    subfolder_file_name = subfolder_file_name.replace(".AVI", "")
                    subfolder_file_name = subfolder_file_name.replace(".jpg", "")

                    # Print the current count
                    print("." * 6, f"Current count: {count}")

                    # If the image already exists the we continue
                    if os.path.exists(f"/content/data/{search_key}/{subfolder_file_name}.png"):
                        print("." * 6, f"{subfolder_file_name} already exists")
                        count += 1
                        continue
                    download_file(f"/content/data/{search_key}", subfolder_file_id, subfolder_file_name)
                    print("." * 6, f"{subfolder_file_name} downloaded")
                    count += 1
                    if count >= limit:
                        break
                if count >= limit:
                        break
            if count >= limit:
                break
        if count >= limit:
            break
    if count >= limit:
        break


.. aves/
...... Current count: 0
...... 2021 04 04 15 15 27 already exists
...... Current count: 1
...... 2021 03 22 10 08 18 already exists
...... Current count: 2
...... 2021 03 22 10 08 01 already exists
.. aves/
...... Current count: 3
...... 2024 02 13 19 00 01 already exists
...... Current count: 4
...... 2024 02 13 19 00 08 already exists
...... Current count: 5
...... 2024 02 13 19 00 07 already exists
...... Current count: 6
...... 2024 02 10 18 59 15 already exists
...... Current count: 7
...... 2024 02 10 18 59 01 already exists
...... Current count: 8
...... 2024 02 10 18 59 14 already exists
...... Current count: 9
...... 2024 02 08 18 54 01 already exists
...... Current count: 10
...... 2024 02 08 18 54 25 already exists
...... Current count: 11
...... 2024 02 08 18 54 24 already exists
...... Current count: 12
...... 2024 02 07 18 54 01 already exists
...... Current count: 13
...... 2024 02 07 18 54 26 already exists
...... Current count: 14
...... 2024 02 07 18 54 25 al